# Sales Rep Agent
A sales rep agent which can send email automatically to person, and can also act as a virtual twin to reply to emails on your behalf.

In [1]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-nano"

## Set Up GMAIL

In [2]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

## Set Up Pushover Notification as a Fallback Strategy

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [7]:
@function_tool
def send_email_notification(subject: str, text_body: str, html_body: str = None):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = "debanjang0.dg@gmail.com"
    msg["Subject"] = subject
    msg.set_content(text_body)
    if html_body:
        msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

    payload = {"user": pushover_user, "token": pushover_token, "message": text_body, "title": subject}
    requests.post(url=pushover_url, data=payload)

In [12]:
send_email_notification("Test Email", "This is a test email notification from the assistant.")

In [10]:
instructions = """
You are an sales agent. You work for a real estate based Saas company RealEstatePro, which provides Saas based solution for small and medium house owner.
Your job is to send email to leads to convince them to use your product. You have to write a convincing email to the lead, highlighting the benefits of using RealEstatePro and how it can help them manage their properties more efficiently. Make sure to personalize the email based on the lead's information and include a call-to-action for them to sign up for a free trial or schedule a demo.

Your email writing style is professional and serious.

Your task:
1. Generate the email.
2. Send the email.
""".strip()

In [16]:
sales_agent = Agent(name="SalesAgent", instructions=instructions, tools=[send_email_notification], model_settings=ModelSettings(model_name=MODEL_NAME))

In [ ]:
message = "Send a sales email"

with trace("Sales email workflow"):
    results = await Runner.run(sales_agent, message)
    print(results.final_output)

with trace("Sales email workflow"):
    results = Runner.run_streamed(sales_agent, message)
    async for event in results.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)

Your sales email has been generated and sent to the lead. Here is a preview of the email:

---
**Subject:** Transform Your Property Management with RealEstatePro

**Body:**

Dear Property Owner,

Managing rental properties can be time-consuming, challenging, and stressful. At RealEstatePro, we understand the daily demands you face as a property owner, which is why we have created a streamlined SaaS solution tailored specifically for small and medium property owners like yourself.

With RealEstatePro, you can:
- Easily manage rental applications, lease agreements, and payments in one secure place
- Automate rent reminders and late fee notifications
- Track maintenance requests and assign tasks seamlessly
- Access real-time financial reports to stay on top of your income and expenses
- Securely store important documents and communicate with tenants effortlessly

Don't let property management tasks overwhelm you. Join hundreds of property owners who have already simplified their workflow 

In [20]:
draw_graph(sales_agent)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH